# SkinIQ — skin type v2 experiment

The first skin type model (v1, `models/skin_type/`) reached only **31.6% test accuracy**: it
memorised the training images by epoch 6 while validation accuracy stayed at 30–35%.
This notebook retrains skin type with settings aimed at that overfitting and compares the two.
The skin condition model (93.1%) is not retrained.

| Setting | v1 | v2 |
|---|---|---|
| Model | EfficientNet-V2-S (20.2M params) | EfficientNet-V2-B0 (5.9M params) — fewer parameters to memorise with |
| Input size | 300 | 224 |
| Learning rate | 3e-4 | 1e-4 |
| Frozen backbone | no | first 3 epochs train only the classifier head |
| Augmentation | mild | strong: crop 50–100%, ±15°, stronger lighting, random erasing |
| Dropout / stochastic depth | 0 / 0 | 0.3 / 0.2 |
| Epochs / patience | 20 / 5 | 25 / 6 |

**Run:** Runtime → T4 GPU, then Runtime → Run all (about 15–25 min). Results go to
`My Drive/SkinIQ/models/skin_type_v2/`; v1 is kept.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/SkinIQ"
SKIN_TYPE_ZIP = "Skin Type Dataset - olimov.zip"
SKIN_CONDITION_ZIP = "Skin Issues Dataset - drishya.zip"
BATCH_SIZE = 64      # lower to 32 if you get "CUDA out of memory"

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
for z in (SKIN_TYPE_ZIP, SKIN_CONDITION_ZIP):
    assert os.path.exists(f"{DRIVE_DIR}/{z}"), f"Not found: {DRIVE_DIR}/{z}"
assert os.path.exists(f"{DRIVE_DIR}/models/skin_type/metrics.json"), "Run train_colab.ipynb first (v1 results are needed for the comparison)"
print("Drive OK")

In [ ]:
%cd /content
!rm -rf SkinIQ && git clone -q https://github.com/rwiie/SkinIQ.git
%cd /content/SkinIQ
!pip install -q "timm>=1.0"
!mkdir -p data/raw
!cp "{DRIVE_DIR}/{SKIN_TYPE_ZIP}" "{DRIVE_DIR}/{SKIN_CONDITION_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_TYPE_ZIP}" -x "__MACOSX/*" -d data/raw
!unzip -q -o "/tmp/{SKIN_CONDITION_ZIP}" -x "__MACOSX/*" -d data/raw
!rm -rf data/raw/skin_type data/raw/skin_condition
!mv "data/raw/Skin Type Dataset - olimov" data/raw/skin_type
!mv "data/raw/Skin Issues Dataset - drishya" data/raw/skin_condition
!python training/prepare_data.py
!git diff --stat training/data_report.md || true

Same split as v1: the last line should print nothing. Now train v2:

In [ ]:
!python training/train.py --task skin_type --run-name skin_type_v2 \
    --model tf_efficientnetv2_b0.in1k --img-size 224 --lr 1e-4 --freeze-epochs 3 \
    --aug strong --drop-rate 0.3 --drop-path-rate 0.2 --epochs 25 --patience 6 \
    --batch-size {BATCH_SIZE} --out-dir "{DRIVE_DIR}/models"

## Compare v1 and v2

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

runs = {"v1": "skin_type", "v2": "skin_type_v2"}
rows, hist = [], {}
for name, folder in runs.items():
    d = f"{DRIVE_DIR}/models/{folder}"
    m = json.load(open(f"{d}/metrics.json"))
    h = pd.read_csv(f"{d}/history.csv"); hist[name] = h
    rows.append({
        "run": name, "model": m["model"], "best epoch": m["best_epoch"],
        "test accuracy": m["test_accuracy"], "macro precision": m["test_macro"]["precision"],
        "macro recall": m["test_macro"]["recall"], "macro F1": m["test_macro"]["f1-score"],
        **{f"F1 {c}": v["f1-score"] for c, v in m["per_class"].items()},
        "train-val loss gap": round(h["val_loss"].iloc[m["best_epoch"] - 1] - h["train_loss"].iloc[m["best_epoch"] - 1], 3),
    })
table = pd.DataFrame(rows).set_index("run").T
display(table)
table.to_csv(f"{DRIVE_DIR}/models/skin_type_v2/comparison_v1_v2.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for (name, h), color in zip(hist.items(), ["#1f77b4", "#ff7f0e"]):
    axes[0].plot(h["epoch"], h["train_loss"], "--", color=color, label=f"{name} train")
    axes[0].plot(h["epoch"], h["val_loss"], color=color, label=f"{name} val")
    axes[1].plot(h["epoch"], h["val_macro_f1"], color=color, label=name)
axes[0].set_title("Loss"); axes[1].set_title("Validation macro-F1"); axes[1].axhline(0.25, color="grey", ls=":", label="chance")
for a in axes: a.set_xlabel("epoch"); a.legend()
fig.tight_layout(); fig.savefig(f"{DRIVE_DIR}/models/skin_type_v2/comparison_v1_v2.png", dpi=150); plt.show()
display(Image(f"{DRIVE_DIR}/models/skin_type_v2/confusion_matrix.png", width=480))

Saved to `My Drive/SkinIQ/models/skin_type_v2/`: the model, metrics, confusion matrix, and
`comparison_v1_v2.csv` / `.png` for the report. Tell Claude when it's done.